# Shear and Moment Diagrams

Common beam configurations from AISC Steel Construction Manual, Table 3-23.
All units are US customary: **kips**, **ft**, **in**.

Loads convention:
- Vertical loads: positive = downward (gravity)
- x-coordinates measured from left end of beam
- Deflection computed with E = 29,000 ksi (steel)

In [ ]:
import matplotlib.pyplot as plt

from civilpy.structural.beam_bending import (
    Beam, CantileverBeam, ProppedCantileverBeam, FixedFixedBeam, ContinuousBeam,
    PointLoadV, DistributedLoadV, PointTorque,
    plot_beam_full,
)
from civilpy.structural.steel import W

E_STEEL = 29000  # ksi

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

---
## Select a Steel Section

Pick a W-shape to get section properties (I, S, Z) used in deflection calculations.

In [ ]:
# Change these to explore different shapes
beam_shape = W('W16X36')

print(f"Shape: W16x36")
print(f"  I_x  = {float(beam_shape.I_x.magnitude):.1f} in⁴")
print(f"  S_x  = {float(beam_shape.S_x.magnitude):.1f} in³")
print(f"  Z_x  = {float(beam_shape.Z_x.magnitude):.1f} in³")
print(f"  d    = {float(beam_shape.depth.magnitude):.3f} in")
print(f"  Self weight = {float(beam_shape.weight.magnitude):.0f} plf")

---
## Case 1: Simple Beam — Uniformly Distributed Load

**AISC SCM Table 3-23, Case 1**

Closed-form checks:
- R = wL/2
- M_max = wL²/8 at midspan
- δ_max = 5wL⁴/(384EI) at midspan

In [ ]:
L = 24.0   # ft  — span
w = 2.0    # kips/ft — uniform dead + live load

beam1 = Beam(L)
beam1.pinned_support = 0
beam1.rolling_support = L
beam1.add_loads([DistributedLoadV(f"{w}", (0, L))])

R_A, R_B = beam1.get_reaction_forces()[1], beam1.get_reaction_forces()[2]
M_max_formula = w * L**2 / 8
delta_max_formula = 5 * w/12 * (L*12)**4 / (384 * E_STEEL * float(beam_shape.I_x.magnitude))  # inches

print(f"Span L = {L} ft,  w = {w} k/ft")
print(f"R_A = R_B = {float(R_A):.2f} kips  (check: {w*L/2:.2f})")
print(f"M_max = {M_max_formula:.2f} kip·ft  at x = L/2 = {L/2:.1f} ft")
print(f"\u03b4_max (W16x36, midspan) = {delta_max_formula:.3f} in  = L/{L*12/delta_max_formula:.0f}")

fig1 = plot_beam_full(beam1, I=beam_shape.I_x, title=f"Case 1: Simple Beam — Uniform Load  w={w} k/ft, L={L} ft")
plt.show()

---
## Case 2: Simple Beam — Triangular Load (Zero at Left, Max at Right)

**AISC SCM Table 3-23, Case 2**

Hydrostatic or soil pressure loading: w(x) = w_max × x / L.

- R_A = w_max·L/6,  R_B = w_max·L/3
- M_max = w_max·L²·(√3/27)  at x = L/√3

<!--
AISC Table 3-23 reference for all cases:
 8  Simple beam — concentrated load at any point
 9  Simple beam — two equal concentrated loads symmetrically placed
10  Simple beam — two equal concentrated loads unsymmetrically placed
11  Simple beam — two unequal concentrated loads unsymmetrically placed
12  Beam fixed at one end, supported at other — uniformly distributed load
13  Beam fixed at one end, supported at other — concentrated load at center
14  Beam fixed at one end, supported at other — concentrated load at any point
15  Beam fixed at both ends — uniformly distributed load
16  Beam fixed at both ends — concentrated load at center
17  Beam fixed at both ends — concentrated load at any point
18  Cantilever — load increasing uniformly to fixed end
19  Cantilever — uniformly distributed load
20  Beam fixed at one end, free to deflect vertically but not rotate at other — UDL
21  Cantilever — concentrated load at any point
22  Cantilever — concentrated load at free end
23  Beam fixed at one end, free to deflect vertically but not rotate — concentrated load
24  Beam overhanging one support — uniformly distributed load
25  Beam overhanging one support — uniformly distributed load on overhang
26  Beam overhanging one support — concentrated load at end of overhang
27  Beam overhanging one support — uniformly distributed load between supports
28  Beam overhanging one support — concentrated load at any point between supports
29  Continuous beam — two equal spans — uniform load on one span
30  Continuous beam — two equal spans — concentrated load at center of one span
31  Continuous beam — two equal spans — concentrated load at any point
32  Beam — uniformly distributed load and variable end moments
33  Beam — concentrated load at center and variable end moments
34  Simple beam — load increasing uniformly from center
35  Simple beam — concentrated moment at end
36  Simple beam — concentrated moment at any point
37  Continuous beam — three equal spans — one end span unloaded
38  Continuous beam — three equal spans — end spans loaded
39  Continuous beam — three equal spans — all spans loaded
40  Continuous beam — four equal spans — third span loaded
41  Continuous beam — four equal spans — first and third spans loaded
42  Continuous beam — four equal spans — all spans loaded
43  Simple beam — one concentrated moving load
44  Simple beam — two equal concentrated moving loads
45  Simple beam — two unequal concentrated moving loads
-->

In [ ]:
L = 20.0      # ft
w_max = 4.0   # kips/ft at right end

beam6 = Beam(L)
beam6.pinned_support = 0
beam6.rolling_support = L
beam6.add_loads([DistributedLoadV(f"{w_max}/{L}*x", (0, L))])

R_A = beam6.get_reaction_forces()[1]
R_B = beam6.get_reaction_forces()[2]
print(f"Triangular load: 0 to {w_max} k/ft over L = {L} ft")
print(f"R_A = {float(R_A):.3f} kips  (check: {w_max*L/6:.3f})")
print(f"R_B = {float(R_B):.3f} kips  (check: {w_max*L/3:.3f})")

fig6 = plot_beam_full(beam6, I=beam_shape.I_x,
                      title=f"Case 2: Triangular Load  0 to {w_max} k/ft, L={L} ft")
plt.show()

---
## Case 3: Simple Beam — Load Increasing Uniformly to Center

**AISC SCM Table 3-23, Case 3**

Symmetric triangular load: zero at both supports, w_max at midspan.

- R_A = R_B = w_max·L/4
- M_max = w_max·L²/12 at midspan

In [ ]:

L = 24.0      # ft
w_max = 3.0   # kips/ft at center (x = L/2)

beamA3 = Beam(L)
beamA3.pinned_support = 0
beamA3.rolling_support = L
beamA3.add_loads([
    DistributedLoadV(f"2*{w_max}/{L}*x", (0, L/2)),
    DistributedLoadV(f"2*{w_max}/{L}*({L}-x)", (L/2, L)),
])

W_total = w_max * L / 2
R_A = beamA3.get_reaction_forces()[1]
M_max_formula = w_max * L**2 / 12

print(f"Load Increasing Uniformly to Center: w_max = {w_max} k/ft at midspan, L = {L} ft")
print(f"Total load W = {W_total:.2f} kips")
print(f"R_A = R_B = {float(R_A):.3f} kips  (check: {W_total/2:.3f})")
print(f"M_max = {M_max_formula:.3f} kip·ft at x = L/2 = {L/2:.1f} ft")

figA3 = plot_beam_full(beamA3, I=beam_shape.I_x,
                       title=f"AISC Case 3: Triangular Load to Center  w_max={w_max} k/ft, L={L} ft")
plt.show()

---
## Case 4: Simple Beam — Uniform Load Partially Distributed

**AISC SCM Table 3-23, Case 4**

Uniform load *w* from x=a to x=a+c.

- R1 = V1 = (wc/2L) * (2L - 2a - c)
- R2 = V2 = (wc/2L) * (2a + c)


In [ ]:
L = 30.0   # ft
w = 2.0    # k/ft
a = 10.0   # ft from left
c = 10.0   # ft length of load

beam4 = Beam(L)
beam4.pinned_support = 0
beam4.rolling_support = L
beam4.add_loads([DistributedLoadV(f"{w}", (a, a+c))])

R1, R2 = beam4.get_reaction_forces()[1], beam4.get_reaction_forces()[2]
check1 = (w*c / (2*L)) * (2*L - 2*a - c)
check2 = (w*c / (2*L)) * (2*a + c)

print(f"Partial uniform load: w={w} k/ft from x={a} to x={a+c}")
print(f"R1 = {float(R1):.3f} kips (check: {check1:.3f})")
print(f"R2 = {float(R2):.3f} kips (check: {check2:.3f})")

fig4 = plot_beam_full(beam4, I=beam_shape.I_x, title='Case 4: Partial Uniform Load')
plt.show()

---
## Case 5: Simple Beam — Uniform Load Partially Distributed at One End

**AISC SCM Table 3-23, Case 5**

Uniform load *w* applied from the left support (x = 0) to x = *a*.

- R_A = w·a·(2L−a)/(2L)
- R_B = w·a²/(2L)
- M_max = R_A²/(2w) at x = R_A/w

In [ ]:
L = 30.0   # ft
w = 2.5    # kips/ft
a = 12.0   # ft — load extends from x=0 to x=a

beamA5 = Beam(L)
beamA5.pinned_support = 0
beamA5.rolling_support = L
beamA5.add_loads([DistributedLoadV(f"{w}", (0, a))])

R_A_check = w * a * (2*L - a) / (2*L)
R_B_check = w * a**2 / (2*L)
x_m = R_A_check / w   # location where V=0, i.e. M_max
M_max_formula = R_A_check**2 / (2*w)

R_A = beamA5.get_reaction_forces()[1]
R_B = beamA5.get_reaction_forces()[2]
print(f"Partial uniform load from x=0 to x={a} ft:  w = {w} k/ft, L = {L} ft")
print(f"R_A = {float(R_A):.3f} kips  (check: {R_A_check:.3f})")
print(f"R_B = {float(R_B):.3f} kips  (check: {R_B_check:.3f})")
print(f"M_max = {M_max_formula:.3f} kip·ft at x = {x_m:.3f} ft")

figA5 = plot_beam_full(beamA5, I=beam_shape.I_x,
                       title=f"AISC Case 5: Partial Uniform at Left End  w={w} k/ft, a={a} ft, L={L} ft")
plt.show()

---
## Case 6: Simple Beam — Uniform Load Partially Distributed at Each End

**AISC SCM Table 3-23, Case 6**

Uniform load *w* applied from each support inward over length *a* (x = 0 to a and x = L−a to L).

- R_A = R_B = w·a
- M_max = w·a²/2 (constant throughout the unloaded middle region)

In [ ]:
L = 30.0   # ft
w = 2.0    # kips/ft at each end
a = 8.0    # ft — loaded length from each support

beamA6 = Beam(L)
beamA6.pinned_support = 0
beamA6.rolling_support = L
beamA6.add_loads([
    DistributedLoadV(f"{w}", (0, a)),
    DistributedLoadV(f"{w}", (L - a, L)),
])

R_check = w * a
M_max_formula = w * a**2 / 2

R_A = beamA6.get_reaction_forces()[1]
print(f"Partial uniform load at each end: w = {w} k/ft, a = {a} ft from each support, L = {L} ft")
print(f"R_A = R_B = {float(R_A):.3f} kips  (check: {R_check:.3f})")
print(f"M_max = {M_max_formula:.3f} kip·ft (constant for {a} ft ≤ x ≤ {L-a} ft)")

figA6 = plot_beam_full(beamA6, I=beam_shape.I_x,
                       title=f"AISC Case 6: Partial Uniform at Each End  w={w} k/ft, a={a} ft, L={L} ft")
plt.show()

---
## Case 7: Simple Beam — Concentrated Load at Midspan

**AISC SCM Table 3-23, Case 7**

AISC Cases 2–6 (triangular and partial distributed loads) are demonstrated above.

Closed-form checks:
- R = P/2
- M_max = PL/4 at midspan
- δ_max = PL³/(48EI)

In [ ]:
L = 24.0   # ft
P = 20.0   # kips

beam2 = Beam(L)
beam2.pinned_support = 0
beam2.rolling_support = L
beam2.add_loads([PointLoadV(P, L/2)])

R_A = beam2.get_reaction_forces()[1]
M_max_formula = P * L / 4
delta_max_formula = P * (L*12)**3 / (48 * E_STEEL * float(beam_shape.I_x.magnitude))  # inches

print(f"P = {P} kips at midspan,  L = {L} ft")
print(f"R_A = R_B = {float(R_A):.2f} kips  (check: {P/2:.2f})")
print(f"M_max = {M_max_formula:.2f} kip·ft")
print(f"\u03b4_max (W16x36) = {delta_max_formula:.3f} in  = L/{L*12/delta_max_formula:.0f}")

fig2 = plot_beam_full(beam2, I=beam_shape.I_x, title=f"Case 7: Concentrated Load at Midspan  P={P} kips, L={L} ft")
plt.show()

---
## Case 8: Simple Beam — Concentrated Load at Any Point

**AISC SCM Table 3-23, Case 8**

Load P at distance *a* from left support, *b = L − a* from right.

- R_A = Pb/L,  R_B = Pa/L
- M_max = Pab/L at x = a
- δ_max at x = √[(L²−b²)/3] when a > b (measured from left; swap a↔b for a < b)

In [ ]:
import math

L = 30.0   # ft
P = 30.0   # kips
a = 18.0   # ft from left support
b = L - a  # ft from right support

beam3 = Beam(L)
beam3.pinned_support = 0
beam3.rolling_support = L
beam3.add_loads([PointLoadV(P, a)])

R_A = beam3.get_reaction_forces()[1]
R_B = beam3.get_reaction_forces()[2]
M_max_formula = P * a * b / L
x_defl_max = math.sqrt((L**2 - b**2) / 3) if a >= b else L - math.sqrt((L**2 - a**2) / 3)

print(f"P = {P} kips at a = {a} ft  (b = {b} ft),  L = {L} ft")
print(f"R_A = {float(R_A):.2f} kips  (check: {P*b/L:.2f})")
print(f"R_B = {float(R_B):.2f} kips  (check: {P*a/L:.2f})")
print(f"M_max = {M_max_formula:.2f} kip·ft at x = {a} ft")
print(f"δ_max at x = {x_defl_max:.3f} ft  (midspan = {L/2:.1f} ft, load at {a} ft)")

fig3 = plot_beam_full(beam3, I=beam_shape.I_x,
                      title=f"Case 8: Concentrated Load at a={a} ft  P={P} kips, L={L} ft")
plt.show()

---
## Case 9: Simple Beam — Two Equal Concentrated Loads, Symmetrically Placed

**AISC SCM Table 3-23, Case 9**

Loads P at distance *a* from each support.

- R = P
- M_max = Pa  (constant between the two loads)
- δ_max = Pa(3L²−4a²)/(24EI)

In [ ]:
L = 30.0   # ft
P = 15.0   # kips each
a = 8.0    # ft from each support

beam4 = Beam(L)
beam4.pinned_support = 0
beam4.rolling_support = L
beam4.add_loads([
    PointLoadV(P, a),
    PointLoadV(P, L - a),
])

M_max_formula = P * a
delta_max_formula = P * (a*12) * (3*(L*12)**2 - 4*(a*12)**2) / (24 * E_STEEL * float(beam_shape.I_x.magnitude))

print(f"2 × P = {P} kips at a = {a} ft from each support,  L = {L} ft")
print(f"R_A = R_B = {P:.2f} kips")
print(f"M_max = {M_max_formula:.2f} kip·ft between loads")
print(f"\u03b4_max (W16x36) = {delta_max_formula:.3f} in  = L/{L*12/delta_max_formula:.0f}")

fig4 = plot_beam_full(beam4, I=beam_shape.I_x,
                      title=f"Case 9: Two Equal Loads  P={P} kips at a={a} ft, L={L} ft")
plt.show()

---
## Case 10: Simple Beam — Two Equal Concentrated Loads Unsymmetrically Placed

**AISC SCM Table 3-23, Case 10**

Two equal loads P at distance a and b from left end (a < b).

- R1 = V1 = P/L * (2L - a - b)
- R2 = V2 = P/L * (a + b)


In [ ]:
L = 20.0
P = 10.0
a = 5.0
b = 12.0

beam10 = Beam(L)
beam10.pinned_support = 0
beam10.rolling_support = L
beam10.add_loads([PointLoadV(P, a), PointLoadV(P, b)])

R1, R2 = beam10.get_reaction_forces()[1], beam10.get_reaction_forces()[2]
check1 = P/L * (2*L - a - b)
check2 = P/L * (a + b)

print(f"Two equal unsymmetric loads: P={P} at x={a} and x={b}")
print(f"R1 = {float(R1):.3f} kips (check: {check1:.3f})")
print(f"R2 = {float(R2):.3f} kips (check: {check2:.3f})")

fig10 = plot_beam_full(beam10, I=beam_shape.I_x, title='Case 10: Two Equal Unsymmetrical Point Loads')
plt.show()

---
## Case 11: Simple Beam — Two Unequal Concentrated Loads Unsymmetrically Placed

**AISC SCM Table 3-23, Case 11**

Loads P1 at a and P2 at b.

- R1 = [P1(L-a) + P2(L-b)] / L
- R2 = [P1*a + P2*b] / L


In [ ]:
from civilpy.structural.aashto.vehicles import HS20Load
L = 50.0  # Increased L to accommodate truck
beam11 = Beam(L)
beam11.pinned_support = 0
beam11.rolling_support = L

hs20 = HS20Load()
for k, axle in hs20.axles.items():
    if isinstance(k, int):
        # Place first axle at some position, e.g., 10ft
        beam11.add_loads([PointLoadV(axle['load_kip'], 10 + axle['dist_ft'])])

R1, R2 = beam11.get_reaction_forces()[1], beam11.get_reaction_forces()[2]

print(f"Case 11: Two unequal loads (Demonstrated with HS-20 Truck Axles)")
print(f"R1 = {float(R1):.3f} kips")
print(f"R2 = {float(R2):.3f} kips")
fig11 = plot_beam_full(beam11, I=beam_shape.I_x, title='Case 11: Two Unequal Loads (HS-20 Axles)')
plt.show()

---
## Case 12: Propped Cantilever — Uniformly Distributed Load

**AISC SCM Table 3-23, Case 12**

- R_A = V_A = 5wL/8
- R_B = V_B (max) = 3wL/8
- M_max = wL²/8 at fixed end
- M_pos = 9wL²/128 at x = 3L/8
- δ_max = wL⁴/(185EI) at x ≈ 0.4215L

In [ ]:
L = 20.0   # ft
w = 1.5    # kips/ft

beam12 = ProppedCantileverBeam(L)
beam12.add_loads([DistributedLoadV(f"{w}", (0, L))])

_, V_A, M_A, R_B = beam12.get_reaction_forces()
M_A_check = w * L**2 / 8
delta_max_formula = (w/12) * (L*12)**4 / (185 * E_STEEL * float(beam_shape.I_x.magnitude))

print(f"Propped Cantilever (Case 12): w = {w} k/ft, L = {L} ft")
print(f"R_A (Fixed) = {V_A:.3f} kips  (check: {5*w*L/8:.3f})")
print(f"R_B (Roller) = {R_B:.3f} kips (check: {3*w*L/8:.3f})")
print(f"M_A (Fixed) = {M_A:.3f} kip·ft (check: {M_A_check:.3f})")
print(f"\u03b4_max (W16x36) ≈ {delta_max_formula:.3f} in")

fig12 = plot_beam_full(beam12, I=beam_shape.I_x,
                       title=f"Case 12: Propped Cantilever — Uniform Load")
plt.show()

---
## Case 13: Propped Cantilever — Concentrated Load at Center

**AISC SCM Table 3-23, Case 13**

- R_A = V_A = 11P/16
- R_B = V_B = 5P/16
- M_max = 3PL/16 at fixed end
- M_pos = 5PL/32 at point of load
- δ_max = PL³/(48EI√5) at x = L√(1/5) ≈ 0.447L

In [ ]:
from civilpy.structural.beam_bending import PointLoadV

L = 24.0   # ft
P = 20.0   # kips at center

beam13 = ProppedCantileverBeam(L)
beam13.add_loads([PointLoadV(P, L/2)])

_, V_A, M_A, R_B = beam13.get_reaction_forces()
M_A_check = 3 * P * L / 16
delta_max_formula = P * (L*12)**3 / (48 * E_STEEL * float(beam_shape.I_x.magnitude) * 5**0.5)

print(f"Propped Cantilever (Case 13): P = {P} kips at center, L = {L} ft")
print(f"R_A (Fixed) = {V_A:.3f} kips  (check: {11*P/16:.3f})")
print(f"R_B (Roller) = {R_B:.3f} kips (check: {5*P/16:.3f})")
print(f"M_A (Fixed) = {M_A:.3f} kip·ft (check: {M_A_check:.3f})")
print(f"\u03b4_max (W16x36) ≈ {delta_max_formula:.3f} in")

fig13 = plot_beam_full(beam13, I=beam_shape.I_x,
                       title=f"Case 13: Propped Cantilever — Load at Center")
plt.show()

---
## Case 14: Propped Cantilever — Concentrated Load at Any Point

**AISC SCM Table 3-23, Case 14**

- R_A = V_A = Pb²(3L - b)/(2L³)
- R_B = V_B = Pa²(3L + a)/(2L³)
- M_A (at fixed end) = Pb(L² - b²)/(2L²)
- M_p (at load) = R_B * b

In [ ]:
L = 20.0   # ft
P = 15.0   # kips
a = 8.0    # ft from fixed end
b = L - a  # ft from roller end

beam14 = ProppedCantileverBeam(L)
beam14.add_loads([PointLoadV(P, a)])

_, V_A, M_A, R_B = beam14.get_reaction_forces()

R_A_check = P * b**2 * (3*L - b) / (2 * L**3)
R_B_check = P * a**2 * (3*L + a) / (2 * L**3)
M_A_check = P * b * (L**2 - b**2) / (2 * L**2)

print(f"Propped Cantilever (Case 14): P = {P} kips at a = {a} ft, L = {L} ft")
print(f"R_A (Fixed) = {V_A:.3f} kips  (check: {R_A_check:.3f})")
print(f"R_B (Roller) = {R_B:.3f} kips (check: {R_B_check:.3f})")
print(f"M_A (Fixed) = {M_A:.3f} kip·ft (check: {M_A_check:.3f})")

fig14 = plot_beam_full(beam14, I=beam_shape.I_x,
                       title=f"Case 14: Propped Cantilever — Load at x={a} ft")
plt.show()

---
## Case 15: Fixed-Fixed Beam — Uniformly Distributed Load

**AISC SCM Table 3-23, Case 15**

- R_A = R_B = wL/2
- M_max = wL²/12 at ends
- M_center = wL²/24
- δ_max = wL⁴/(384EI) at center

In [ ]:
from civilpy.structural.beam_bending import FixedFixedBeam

L = 30.0   # ft
w = 2.0    # kips/ft

beam15 = FixedFixedBeam(L)
beam15.add_loads([DistributedLoadV(f"{w}", (0, L))])

_, V_A, M_A, V_B, M_B = beam15.get_reaction_forces()
M_end_check = w * L**2 / 12
delta_max_formula = (w/12) * (L*12)**4 / (384 * E_STEEL * float(beam_shape.I_x.magnitude))

print(f"Fixed-Fixed Beam (Case 15): w = {w} k/ft, L = {L} ft")
print(f"R_A = R_B = {V_A:.2f} kips  (check: {w*L/2:.2f})")
print(f"M_A = M_B = {M_A:.2f} kip·ft (check: {M_end_check:.2f})")
print(f"\u03b4_max (W16x36) = {delta_max_formula:.3f} in")

fig15 = plot_beam_full(beam15, I=beam_shape.I_x,
                       title=f"Case 15: Fixed-Fixed Beam — Uniform Load")
plt.show()

---
## Case 16: Fixed-Fixed Beam — Concentrated Load at Center

**AISC SCM Table 3-23, Case 16**

- R_A = R_B = P/2
- M_max = PL/8 at ends and at center
- δ_max = PL³/(192EI) at center

In [ ]:
L = 20.0   # ft
P = 30.0   # kips

beam16 = FixedFixedBeam(L)
beam16.add_loads([PointLoadV(P, L/2)])

_, V_A, M_A, V_B, M_B = beam16.get_reaction_forces()
M_check = P * L / 8
delta_max_formula = P * (L*12)**3 / (192 * E_STEEL * float(beam_shape.I_x.magnitude))

print(f"Fixed-Fixed Beam (Case 16): P = {P} kips at center, L = {L} ft")
print(f"R_A = R_B = {V_A:.2f} kips  (check: {P/2:.2f})")
print(f"M_A = M_B = {M_A:.2f} kip·ft (check: {M_check:.2f})")
print(f"\u03b4_max (W16x36) = {delta_max_formula:.3f} in")

fig16 = plot_beam_full(beam16, I=beam_shape.I_x,
                       title=f"Case 16: Fixed-Fixed Beam — Load at Center")
plt.show()

---
## Case 17: Fixed-Fixed Beam — Concentrated Load at Any Point

**AISC SCM Table 3-23, Case 17**

- R_A = V_A = Pb²(3a + b)/L³
- R_B = V_B = Pa²(a + 3b)/L³
- M_A (at left fixed end) = Pab²/L²
- M_B (at right fixed end) = Pa²b/L²
- δ_max = 2Pa³b²/(3EI(3a+b)²) at x = 2aL/(3a+b) (for a > b)

In [ ]:
L = 25.0   # ft
P = 25.0   # kips
a = 10.0   # ft from left
b = L - a  # ft from right

beam17 = FixedFixedBeam(L)
beam17.add_loads([PointLoadV(P, a)])

_, V_A, M_A, V_B, M_B = beam17.get_reaction_forces()

R_A_check = P * b**2 * (3*a + b) / (L**3)
R_B_check = P * a**2 * (a + 3*b) / (L**3)
M_A_check = P * a * b**2 / (L**2)
M_B_check = P * a**2 * b / (L**2)

print(f"Fixed-Fixed Beam (Case 17): P = {P} kips at a = {a} ft, L = {L} ft")
print(f"R_A = {V_A:.3f} kips  (check: {R_A_check:.3f})")
print(f"R_B = {V_B:.3f} kips  (check: {R_B_check:.3f})")
print(f"M_A = {M_A:.3f} kip·ft (check: {M_A_check:.3f})")
print(f"M_B = {M_B:.3f} kip·ft (check: {M_B_check:.3f})")

fig17 = plot_beam_full(beam17, I=beam_shape.I_x,
                       title=f"Case 17: Fixed-Fixed Beam — Load at x={a} ft")
plt.show()

---
## Case 18: Cantilever — Load Increasing Uniformly to Fixed End

**AISC SCM Table 3-23, Case 18**

- R = V = W = wL/2
- M_max = WL/3 = wL²/6 at fixed end
- δ_max = wL⁴/(30EI) at free end

In [ ]:
L = 15.0   # ft
w = 3.0    # kips/ft at fixed end

cant18 = CantileverBeam(L)
cant18.add_loads([DistributedLoadV(f"{w}/{L}*x", (0, L))])

V_A, M_A = cant18.get_reaction_forces()
M_max_formula = w * L**2 / 6
delta_free = (w/12) * (L*12)**4 / (30 * E_STEEL * float(beam_shape.I_x.magnitude))

print(f"Cantilever (Case 18): w = {w} k/ft at fixed end, L = {L} ft")
print(f"V_A = {V_A:.2f} kips  (check: {w*L/2:.2f})")
print(f"M_A = {M_A:.2f} kip·ft  (check: {M_max_formula:.2f})")
print(f"\u03b4_free (W16x36) = {delta_free:.3f} in")

fig18 = plot_beam_full(cant18, I=beam_shape.I_x,
                       title=f"Case 18: Cantilever — Load Increasing to Fixed End  w={w} k/ft, L={L} ft")
plt.show()

---
## Case 19: Cantilever — Uniform Load over Full Span

**AISC SCM Table 3-23, Case 19**

- V_A = wL,  M_A = wL²/2
- δ_max = wL⁴/(8EI) at free end

In [ ]:
L = 12.0   # ft
w = 2.0    # kips/ft

cant2 = CantileverBeam(L)
cant2.add_loads([DistributedLoadV(f"{w}", (0, L))])

V_A, M_A = cant2.get_reaction_forces()
M_max_formula = w * L**2 / 2
delta_free = (w/12) * (L*12)**4 / (8 * E_STEEL * float(beam_shape.I_x.magnitude))

print(f"Cantilever: w = {w} k/ft,  L = {L} ft")
print(f"V_A = {V_A:.2f} kips  (check: {w*L:.2f})")
print(f"M_A = {M_A:.2f} kip·ft  (check: {M_max_formula:.2f})")
print(f"\u03b4_free (W16x36) = {delta_free:.3f} in")

fig9 = plot_beam_full(cant2, I=beam_shape.I_x,
                      title=f"Case 19: Cantilever — Uniform Load  w={w} k/ft, L={L} ft")
plt.show()

---
## Superposition Example 1: Simple Beam — Uniform Load + Concentrated Load at Midspan

Combined loading common for floor beams carrying a column point load in addition to uniform dead/live load.

By superposition:
- M_max = PL/4 + wL²/8

In [ ]:
L = 30.0   # ft
w = 1.5    # kips/ft
P = 25.0   # kips at midspan

beam10 = Beam(L)
beam10.pinned_support = 0
beam10.rolling_support = L
beam10.add_loads([
    DistributedLoadV(f"{w}", (0, L)),
    PointLoadV(P, L/2),
])

R_A = beam10.get_reaction_forces()[1]
M_max_formula = P * L / 4 + w * L**2 / 8

print(f"w = {w} k/ft + P = {P} kips at midspan,  L = {L} ft")
print(f"R_A = R_B = {float(R_A):.2f} kips  (check: {(w*L + P)/2:.2f})")
print(f"M_max (superposition) = {M_max_formula:.2f} kip·ft")

fig10 = plot_beam_full(beam10, I=beam_shape.I_x,
                       title=f"Superposition Example 1: Uniform + Midspan Point Load  w={w} k/ft, P={P} kips, L={L} ft")
plt.show()

---
## Superposition Example 2: Simple Beam — Three Equal Concentrated Loads, Equally Spaced

Three loads P at L/4, L/2, and 3L/4.

- R_A = R_B = 3P/2
- M_max = 3PL/8 between center and outer loads

In [ ]:
L = 40.0   # ft
P = 10.0   # kips each

beam11 = Beam(L)
beam11.pinned_support = 0
beam11.rolling_support = L
beam11.add_loads([
    PointLoadV(P, L/4),
    PointLoadV(P, L/2),
    PointLoadV(P, 3*L/4),
])

R_A = beam11.get_reaction_forces()[1]
print(f"3 × P = {P} kips at L/4, L/2, 3L/4  —  L = {L} ft")
print(f"R_A = R_B = {float(R_A):.2f} kips  (check: {3*P/2:.2f})")
print(f"M_max = {float(R_A) * L/4:.2f} kip·ft")

fig11 = plot_beam_full(beam11, I=beam_shape.I_x,
                       title=f"Superposition Example 2: Three Equal Loads at L/4 Pts  P={P} kips each, L={L} ft")
plt.show()

---
## Parametric Study: Beam Size vs. Deflection Limit

Find the minimum W-shape satisfying L/360 live-load deflection for a simple beam with uniform live load.
Demonstrates using the steel module to iterate over sections.

In [ ]:
L = 30.0         # ft
w_LL = 1.5       # kips/ft live load
defl_limit = L * 12 / 360  # inches

# W-shapes to check (lightest first, all from W16 family)
candidates = ['W16X26', 'W16X31', 'W16X36', 'W16X40', 'W16X45', 'W16X50']

print(f"L = {L} ft,  w_LL = {w_LL} k/ft,  deflection limit = L/360 = {defl_limit:.3f} in")
print(f"{'Shape':<12} {'I_x (in⁴)':<14} {'δ_max (in)':<14} {'L/δ':<10} {'OK?'}")
print("-" * 60)

for shape_label in candidates:
    shape_obj = W(shape_label)
    I_val = float(shape_obj.I_x.magnitude)
    delta = 5 * (w_LL/12) * (L*12)**4 / (384 * E_STEEL * I_val)
    ratio = L * 12 / delta
    ok = "✓" if delta <= defl_limit else "✗"
    print(f"{shape_label:<12} {I_val:<14.1f} {delta:<14.4f} {ratio:<10.0f} {ok}")

---
## Custom Configuration

Template cell: build any beam configuration by composing load objects.

In [ ]:
# --- USER INPUT ---
L    = 40.0   # span, ft
shape_label = 'W21X50'

custom_beam = Beam(L)
custom_beam.pinned_support = 0
custom_beam.rolling_support = L

custom_beam.add_loads([
    DistributedLoadV("2.0", (0, L)),      # 2 k/ft full span
    PointLoadV(15, 10),                   # 15 kip at x=10 ft
    PointLoadV(20, 25),                   # 20 kip at x=25 ft
])

custom_shape = W(shape_label)
_, R_A, R_B = custom_beam.get_reaction_forces()
print(f"R_A = {float(R_A):.2f} kips,  R_B = {float(R_B):.2f} kips")

fig_custom = plot_beam_full(custom_beam, I=custom_shape.I_x, title=f"Custom: {shape_label}, L={L} ft")
plt.show()

---
## Case 20: Beam Fixed at One End, Free to Deflect Vertically but Not Rotate at Other — Uniformly Distributed Load

**AISC SCM Table 3-23, Case 20**

- R1 = V1 = wL
- M1 = wL^2 / 3
- M2 = wL^2 / 6


In [ ]:
L = 15.0
w = 2.0

# This is a Fixed-Fixed beam where one end is on a vertical slider (zero shear if no load, but here there is load)
# Actually, it can be modeled as a FixedFixedBeam with the symmetry/boundary conditions handled by sympy solver
# But civilpy's FixedFixedBeam assumes both ends are fully fixed (delta=0, theta=0)
# Case 20 is different. theta=0 at both ends, but delta is allowed at one end.
# For now, let's use ProppedCantilever and adjust, or just implement as requested.
from civilpy.structural.beam_bending import FixedFixedBeam

# Case 20 is actually half of a fixed-fixed beam with symmetric load.
beam20 = FixedFixedBeam(2*L)
beam20.add_loads([DistributedLoadV(f"{w}", (0, 2*L))])

reactions = beam20.get_reaction_forces()
R1 = reactions[1]
M1 = reactions[2]
M2 = beam20.get_moment_function().subs('x', L)

print(f"Case 20: w={w}, L={L}")
print(f"R1 = {float(R1):.3f} (check: {w*L:.3f})")

fig20 = plot_beam_full(beam20, I=beam_shape.I_x, title='Case 20 (Modeled as part of Fixed-Fixed)')
plt.show()

---
## Case 21: Cantilever — Concentrated Load at Any Point

**AISC SCM Table 3-23, Case 21**

- R = V = P
- M_max = Pa at fixed end
- δ_max = Pa²(3L - a)/(6EI) at free end

In [ ]:
L = 20.0   # ft
P = 15.0   # kips
a = 12.0   # ft from fixed end

cant21 = CantileverBeam(L)
cant21.add_loads([PointLoadV(P, a)])

V_A, M_A = cant21.get_reaction_forces()
M_max_formula = P * a
delta_free = P * (a*12)**2 * (3*L*12 - a*12) / (6 * E_STEEL * float(beam_shape.I_x.magnitude))

print(f"Cantilever (Case 21): P = {P} kips at a = {a} ft, L = {L} ft")
print(f"V_A = {V_A:.2f} kips  (check: {P:.2f})")
print(f"M_A = {M_A:.2f} kip·ft  (check: {M_max_formula:.2f})")
print(f"\u03b4_free (W16x36) = {delta_free:.3f} in")

fig21 = plot_beam_full(cant21, I=beam_shape.I_x,
                       title=f"Case 21: Cantilever — Load at x={a} ft  P={P} kips, L={L} ft")
plt.show()

---
## Case 22: Cantilever — Concentrated Load at Free End

**AISC SCM Table 3-23, Case 22**

- V = P (constant)
- M_max = PL at fixed end
- δ_max = PL³/(3EI) at free end

In [ ]:
L = 12.0   # ft
P = 10.0   # kips at free end

cant1 = CantileverBeam(L)
cant1.add_loads([PointLoadV(P, L)])

V_A, M_A = cant1.get_reaction_forces()
M_max_formula = P * L
delta_free = P * (L*12)**3 / (3 * E_STEEL * float(beam_shape.I_x.magnitude))

print(f"Cantilever: P = {P} kips at free end,  L = {L} ft")
print(f"Reaction: V_A = {V_A:.2f} kips  (check: {P:.2f})")
print(f"Fixed moment: M_A = {M_A:.2f} kip·ft  (check: {M_max_formula:.2f})")
print(f"\u03b4_free (W16x36) = {delta_free:.3f} in")

fig8 = plot_beam_full(cant1, I=beam_shape.I_x,
                      title=f"Case 22: Cantilever — Concentrated Load at Free End  P={P} kips, L={L} ft")
plt.show()

---
## Case 23: Beam Fixed at One End, Free to Deflect Vertically but Not Rotate — Concentrated Load at Free End

**AISC SCM Table 3-23, Case 23**

- R1 = V1 = P
- M1 = M2 = PL / 2


In [ ]:
L = 10.0
P = 5.0

beam23 = FixedFixedBeam(2*L)
beam23.add_loads([PointLoadV(P, L)])

reactions = beam23.get_reaction_forces()
R1 = reactions[1]
M1 = reactions[2]
M2 = beam23.get_moment_function().subs('x', L)

print(f"Case 23: P={P}, L={L}")
print(f"R1 = {float(R1):.3f} (check: {P:.3f})")
print(f"M1 = {float(M1):.3f} (check: {P*L/2:.3f})")

fig23 = plot_beam_full(beam23, I=beam_shape.I_x, title='Case 23 (Modeled as part of Fixed-Fixed)')
plt.show()

---
## Case 24: Beam Overhanging One Support — Uniform Load, Full Span

**AISC SCM Table 3-23, Case 24**

Beam from 0 to L+a, pin at A (x=0), roller at B (x=L), overhang length a.
Uniform load *w* (kips/ft) over the full beam length L+a.

- R_A = w(L²−a²)/(2L)
- R_B = w(L+a)²/(2L)
- M_B = −wa²/2  (hogging at interior support B)
- V = 0 at x = (L²−a²)/(2L) between supports (when R_A > 0)
- M_max+ = R_A²/(2w)  at that location

In [ ]:
L = 20.0   # ft — span between supports
a = 6.0    # ft — overhang length
w24 = 2.0  # kips/ft — uniform load over full beam

total24 = L + a
beam24 = Beam(total24)
beam24.pinned_support = 0
beam24.rolling_support = L
beam24.add_loads([DistributedLoadV(f"{w24}", (0, total24))])

R_A24_check = w24 * (L**2 - a**2) / (2 * L)
R_B24_check = w24 * (L + a)**2 / (2 * L)
M_B24 = -w24 * a**2 / 2

R_A24 = float(beam24.get_reaction_forces()[1])
R_B24 = float(beam24.get_reaction_forces()[2])
print(f"L = {L} ft,  a = {a} ft,  w = {w24} k/ft over full beam [0, {total24}] ft")
print(f"R_A = {R_A24:.3f} kips  (check: {R_A24_check:.3f})")
print(f"R_B = {R_B24:.3f} kips  (check: {R_B24_check:.3f})")
print(f"M_B = {M_B24:.3f} kip·ft  (hogging at interior support B)")
if R_A24_check > 0:
    x_zero_V24 = R_A24_check / w24
    M_max24_pos = R_A24_check**2 / (2 * w24)
    print(f"M_max+ = {M_max24_pos:.3f} kip·ft  at x = {x_zero_V24:.3f} ft")

fig24 = plot_beam_full(beam24, I=beam_shape.I_x,
                       title=f"Case 24: Overhang — UDL Full Span  w={w24} k/ft, L={L} ft, a={a} ft")
plt.show()

---
## Case 25: Beam Overhanging One Support — Uniform Load on Overhang Only

**AISC SCM Table 3-23, Case 25**

Beam from 0 to L+a, pin at A (x=0), roller at B (x=L).
Uniform load *w* (kips/ft) over the overhang [L, L+a] only; span [0, L] is unloaded.

- R_A = −wa²/(2L)  (downward)
- R_B = wa(2L+a)/(2L)  (upward)
- M_B = −wa²/2  (hogging at interior support B)
- Shear between supports is constant = R_A; moment varies linearly 0 at A → −wa²/2 at B

In [ ]:
L = 20.0   # ft — span between supports
a = 6.0    # ft — overhang length
w25 = 2.0  # kips/ft — uniform load on overhang only

total25 = L + a
beam25 = Beam(total25)
beam25.pinned_support = 0
beam25.rolling_support = L
beam25.add_loads([DistributedLoadV(f"{w25}", (L, total25))])

R_A25_check = -w25 * a**2 / (2 * L)             # downward
R_B25_check = w25 * a * (2 * L + a) / (2 * L)
M_B25 = -w25 * a**2 / 2

R_A25 = float(beam25.get_reaction_forces()[1])
R_B25 = float(beam25.get_reaction_forces()[2])
print(f"L = {L} ft,  a = {a} ft,  w = {w25} k/ft on overhang [{L}, {total25}] ft")
print(f"R_A = {R_A25:.3f} kips  (check: {R_A25_check:.3f},  downward if negative)")
print(f"R_B = {R_B25:.3f} kips  (check: {R_B25_check:.3f})")
print(f"M_B = {M_B25:.3f} kip·ft  (hogging at interior support B)")

fig25 = plot_beam_full(beam25, I=beam_shape.I_x,
                       title=f"Case 25: Overhang — UDL on Overhang Only  w={w25} k/ft, L={L} ft, a={a} ft")
plt.show()

---
## Case 26: Beam Overhanging One Support — Concentrated Load at End of Overhang

**AISC SCM Table 3-23, Case 26**

Beam from 0 to L+a, pin at A (x=0), roller at B (x=L).
Concentrated load P at free end x = L+a.

- R_A = −Pa/L  (downward when a > 0)
- R_B = P(L+a)/L  (upward)
- M_B = −Pa  (hogging at interior support B)
- Shear between supports is constant = R_A; moment varies linearly 0 → −Pa

In [ ]:
L = 20.0   # ft — span between supports
a = 6.0    # ft — overhang length
P26 = 15.0   # kips — concentrated load at free end

total26 = L + a
beam26 = Beam(total26)
beam26.pinned_support = 0
beam26.rolling_support = L
beam26.add_loads([PointLoadV(P26, total26)])

R_A26_check = -P26 * a / L           # downward
R_B26_check = P26 * (L + a) / L
M_B26 = -P26 * a

R_A26 = float(beam26.get_reaction_forces()[1])
R_B26 = float(beam26.get_reaction_forces()[2])
print(f"L = {L} ft,  a = {a} ft,  P = {P26} kips at free end (x = {total26} ft)")
print(f"R_A = {R_A26:.3f} kips  (check: {R_A26_check:.3f},  downward if negative)")
print(f"R_B = {R_B26:.3f} kips  (check: {R_B26_check:.3f})")
print(f"M_B = {M_B26:.3f} kip·ft  (hogging at interior support B)")

fig26 = plot_beam_full(beam26, I=beam_shape.I_x,
                       title=f"Case 26: Overhang — Conc. Load at Free End  P={P26} kips, L={L} ft, a={a} ft")
plt.show()

---
## Case 27: Beam Overhanging One Support — Uniform Load Between Supports Only

**AISC SCM Table 3-23, Case 27**

Beam from 0 to L+a, pin at A (x=0), roller at B (x=L).
Uniform load *w* (kips/ft) over span [0, L] only; overhang [L, L+a] is unloaded.

- R_A = R_B = wL/2
- M_max = wL²/8  at x = L/2  (sagging)
- Zero shear and zero moment throughout the overhang

In [ ]:
L = 20.0   # ft — span between supports
a = 6.0    # ft — overhang length (unloaded)
w27 = 2.0  # kips/ft — uniform load between supports only

total27 = L + a
beam27 = Beam(total27)
beam27.pinned_support = 0
beam27.rolling_support = L
beam27.add_loads([DistributedLoadV(f"{w27}", (0, L))])

R_A27_check = w27 * L / 2
R_B27_check = w27 * L / 2
M_max27_check = w27 * L**2 / 8

R_A27 = float(beam27.get_reaction_forces()[1])
R_B27 = float(beam27.get_reaction_forces()[2])
print(f"L = {L} ft,  a = {a} ft (unloaded overhang),  w = {w27} k/ft between supports")
print(f"R_A = {R_A27:.3f} kips  (check: {R_A27_check:.3f})")
print(f"R_B = {R_B27:.3f} kips  (check: {R_B27_check:.3f})")
print(f"M_max = {M_max27_check:.3f} kip·ft  at x = L/2 = {L/2:.1f} ft")
print(f"Overhang [{L}–{total27} ft]: V = 0, M = 0")

fig27 = plot_beam_full(beam27, I=beam_shape.I_x,
                       title=f"Case 27: Overhang — UDL Between Supports  w={w27} k/ft, L={L} ft, a={a} ft")
plt.show()

---
## Case 28: Beam Overhanging One Support — Concentrated Load at Any Point Between Supports

**AISC SCM Table 3-23, Case 28**

Beam from 0 to L+a, pin at A (x=0), roller at B (x=L).
Concentrated load P at x = c (0 < c < L); overhang [L, L+a] is unloaded.

- R_A = P(L−c)/L
- R_B = Pc/L
- M_max = Pc(L−c)/L at x = c  (sagging)
- Zero shear and zero moment throughout the overhang

In [ ]:
L = 20.0   # ft — span between supports
a = 6.0    # ft — overhang length (unloaded)
P28 = 15.0   # kips — concentrated load between supports
c28 = 8.0    # ft — distance from A to load

total28 = L + a
beam28 = Beam(total28)
beam28.pinned_support = 0
beam28.rolling_support = L
beam28.add_loads([PointLoadV(P28, c28)])

R_A28_check = P28 * (L - c28) / L
R_B28_check = P28 * c28 / L
M_max28_check = P28 * c28 * (L - c28) / L

R_A28 = float(beam28.get_reaction_forces()[1])
R_B28 = float(beam28.get_reaction_forces()[2])
print(f"L = {L} ft,  a = {a} ft (unloaded overhang),  P = {P28} kips at c = {c28} ft")
print(f"R_A = {R_A28:.3f} kips  (check: {R_A28_check:.3f})")
print(f"R_B = {R_B28:.3f} kips  (check: {R_B28_check:.3f})")
print(f"M_max = {M_max28_check:.3f} kip·ft  at x = {c28} ft")
print(f"Overhang [{L}–{total28} ft]: V = 0, M = 0")

fig28 = plot_beam_full(beam28, I=beam_shape.I_x,
                       title=f"Case 28: Overhang — Conc. Load Between Supports  P={P28} kips at c={c28} ft, L={L} ft, a={a} ft")
plt.show()

---
## Case 29: Continuous Beam — Two Equal Spans — Uniform Load on One Span

**AISC SCM Table 3-23, Case 29**

Closed-form checks:
- R1 = 7wL/16
- R2 = 5wL/8
- R3 = -wL/16
- M_max = 49wL²/512 at x = 7L/16
- M_2 = wL²/16
- δ_max = 0.0092 wL⁴/EI at x = 0.446L

In [ ]:
L = 20.0  # Span length (ft)
w = 1.0   # kips/ft

beam29 = ContinuousBeam(2*L, intermediate_supports=[L])
beam29.pinned_support = 0
beam29.rolling_support = 2*L
beam29.add_loads([DistributedLoadV(f'{w}', (0, L))])

reactions = beam29.get_reaction_forces()
R1 = reactions[1]
R2 = beam29._redundant_reactions[L]
R3 = reactions[2]

print(f"Reactions: R1={float(R1):.3f}, R2={float(R2):.3f}, R3={float(R3):.3f}")
print(f"Checks: R1={7*w*L/16:.3f}, R2={5*w*L/8:.3f}, R3={-w*L/16:.3f}")

fig29 = plot_beam_full(beam29, I=beam_shape.I_x, title='Case 29: Continuous Beam - Two Equal Spans - Uniform Load on One Span')
plt.show()

---
## Case 30: Continuous Beam — Two Equal Spans — Concentrated Load at Center of One Span

**AISC SCM Table 3-23, Case 30**

Closed-form checks:
- R1 = 13P/32
- R2 = 11P/16
- R3 = -3P/32
- M_max = 13PL/64 at point of load
- M_2 = 3PL/32

In [ ]:
L = 20.0
P = 10.0

beam30 = ContinuousBeam(2*L, intermediate_supports=[L])
beam30.pinned_support = 0
beam30.rolling_support = 2*L
beam30.add_loads([PointLoadV(P, L/2)])

reactions = beam30.get_reaction_forces()
R1, R3 = reactions[1], reactions[2]
R2 = beam30._redundant_reactions[L]

print(f"Reactions: R1={float(R1):.3f}, R2={float(R2):.3f}, R3={float(R3):.3f}")
print(f"Checks: R1={13*P/32:.3f}, R2={11*P/16:.3f}, R3={-3*P/32:.3f}")

fig30 = plot_beam_full(beam30, I=beam_shape.I_x, title='Case 30: Continuous Beam - Two Equal Spans - Concentrated Load at Center of One Span')
plt.show()

---
## Case 31: Continuous Beam — Two Equal Spans — Concentrated Load at Any Point

**AISC SCM Table 3-23, Case 31**

Closed-form checks (load P at distance a from R1, b from R2):
- R1 = Pa/2L² * [4L² - a(L+a)]  <- No, wait, AISC formula is different.
Actually AISC SCM 15th Ed:
- R1 = Pb/(4L³) * [4L² - a(L+a)]
- R2 = Pa/(2L³) * [3L² - a²]
- R3 = -Pab/(4L³) * [L+a]
- M_2 = Pab/(4L²) * [L+a]

In [ ]:
L = 20.0
a = 8.0
b = L - a
P = 10.0

beam31 = ContinuousBeam(2*L, intermediate_supports=[L])
beam31.pinned_support = 0
beam31.rolling_support = 2*L
beam31.add_loads([PointLoadV(P, a)])

reactions = beam31.get_reaction_forces()
R1, R3 = reactions[1], reactions[2]
R2 = beam31._redundant_reactions[L]

check1 = P*b/(4*L**3) * (4*L**2 - a*(L+a))
check2 = P*a/(2*L**3) * (3*L**2 - a**2)
check3 = -P*a*b/(4*L**3) * (L+a)

print(f"Reactions: R1={float(R1):.3f}, R2={float(R2):.3f}, R3={float(R3):.3f}")
print(f"Checks: R1={check1:.3f}, R2={check2:.3f}, R3={check3:.3f}")

fig31 = plot_beam_full(beam31, I=beam_shape.I_x, title='Case 31: Continuous Beam - Two Equal Spans - Concentrated Load at Any Point')
plt.show()

---
## Case 32: Simple Beam — Uniformly Distributed Load and Variable End Moments

**AISC SCM Table 3-23, Case 32**

- R_A = wL/2 + (M1 - M2)/L
- R_B = wL/2 - (M1 - M2)/L
- M(x) = M1 + R_A*x - wx²/2

In [ ]:
L = 25.0    # ft
w = 1.2     # kips/ft
M1 = 50.0   # kip·ft (left end, clockwise)
M2 = -30.0  # kip·ft (right end, clockwise; negative is CCW)

beam32 = Beam(L)
beam32.pinned_support = 0
beam32.rolling_support = L
beam32.add_loads([
    DistributedLoadV(f"{w}", (0, L)),
    PointTorque(M1, 0.001),
    PointTorque(M2, L - 0.001)
])

R_A = beam32.get_reaction_forces()[1]
R_B = beam32.get_reaction_forces()[2]

print(f"Case 32: w = {w} k/ft, M1 = {M1} k-ft, M2 = {M2} k-ft, L = {L} ft")
print(f"R_A = {float(R_A):.2f} kips  (check: {w*L/2 + (M1 - M2)/L:.2f})")
print(f"R_B = {float(R_B):.2f} kips  (check: {w*L/2 - (M1 - M2)/L:.2f})")

fig32 = plot_beam_full(beam32, I=beam_shape.I_x,
                       title=f"Case 32: UDL and Variable End Moments")
plt.show()

---
## Case 33: Simple Beam — Concentrated Load at Center and Variable End Moments

**AISC SCM Table 3-23, Case 33**

- R_A = P/2 + (M1 - M2)/L
- R_B = P/2 - (M1 - M2)/L

In [ ]:
L = 20.0    # ft
P = 20.0    # kips at center
M1 = 40.0   # kip·ft (left end)
M2 = -20.0  # kip·ft (right end)

beam33 = Beam(L)
beam33.pinned_support = 0
beam33.rolling_support = L
beam33.add_loads([
    PointLoadV(P, L/2),
    PointTorque(M1, 0.001),
    PointTorque(M2, L - 0.001)
])

R_A = beam33.get_reaction_forces()[1]
R_B = beam33.get_reaction_forces()[2]

print(f"Case 33: P = {P} kips, M1 = {M1} k-ft, M2 = {M2} k-ft, L = {L} ft")
print(f"R_A = {float(R_A):.2f} kips  (check: {P/2 + (M1 - M2)/L:.2f})")
print(f"R_B = {float(R_B):.2f} kips  (check: {P/2 - (M1 - M2)/L:.2f})")

fig33 = plot_beam_full(beam33, I=beam_shape.I_x,
                       title=f"Case 33: Point Load and Variable End Moments")
plt.show()

---
## Case 34: Simple Beam — Load Increasing Uniformly from Center

**AISC SCM Table 3-23, Case 34**

- R_A = R_B = W/2 = wL/4
- M_max = wL²/24 at midspan
- δ_max = wL⁴/(60EI) at midspan

In [ ]:
L = 30.0    # ft
w = 2.5     # kips/ft at ends

beam34 = Beam(L)
beam34.pinned_support = 0
beam34.rolling_support = L
beam34.add_loads([
    DistributedLoadV(f"2*{w}/{L}*({L}/2-x)", (0, L/2)),
    DistributedLoadV(f"2*{w}/{L}*(x-{L}/2)", (L/2, L))
])

R_A = beam34.get_reaction_forces()[1]
M_max_formula = w * L**2 / 24
delta_max_formula = (w/12) * (L*12)**4 / (60 * E_STEEL * float(beam_shape.I_x.magnitude))

print(f"Case 34: w = {w} k/ft at ends, L = {L} ft")
print(f"R_A = R_B = {float(R_A):.2f} kips  (check: {w*L/4:.2f})")
print(f"M_max = {M_max_formula:.2f} kip·ft at midspan")
print(f"\u03b4_max (W16x36) = {delta_max_formula:.3f} in")

fig34 = plot_beam_full(beam34, I=beam_shape.I_x,
                       title=f"Case 34: Load Increasing to Ends  w={w} k/ft, L={L} ft")
plt.show()

---
## Case 35: Simple Beam — Concentrated Moment at One End

**AISC SCM Table 3-23, Case 35**

Clockwise moment M₀ applied at left end (x = 0).

- R_A = −M₀/L (downward),  R_B = +M₀/L (upward)
- M(x) = M₀(1 − x/L)  (linear, max at left end, zero at right)
- δ_max at x = L/√3 ≈ 0.577L

In [ ]:
L = 20.0    # ft
M0 = 100.0  # kip·ft, applied clockwise at left end

beam7 = Beam(L)
beam7.pinned_support = 0
beam7.rolling_support = L
beam7.add_loads([PointTorque(M0, 0.001)])  # applied at left (x≈0 to avoid zero-point instability)

R_A = beam7.get_reaction_forces()[1]
R_B = beam7.get_reaction_forces()[2]
# CW moment at left end: R_A = -M0/L (downward), R_B = +M0/L (upward)
print(f"Applied moment M₀ = {M0} kip·ft clockwise at left end,  L = {L} ft")
print(f"R_A = {float(R_A):.3f} kips  (check: {-M0/L:.3f}, downward)")
print(f"R_B = {float(R_B):.3f} kips  (check: {M0/L:.3f}, upward)")

fig7 = plot_beam_full(beam7, I=beam_shape.I_x,
                      title=f"Case 35: Moment at Left End  M₀={M0} kip·ft, L={L} ft")
plt.show()

---
## Case 36: Simple Beam — Concentrated Moment at Any Point

**AISC SCM Table 3-23, Case 36**

- R_A = M/L (downward if M is CW)
- R_B = M/L (upward if M is CW)

In [ ]:
L = 20.0    # ft
M = 80.0    # kip·ft clockwise
a = 12.0    # ft from left end

beam36 = Beam(L)
beam36.pinned_support = 0
beam36.rolling_support = L
beam36.add_loads([PointTorque(M, a)])

R_A = beam36.get_reaction_forces()[1]
R_B = beam36.get_reaction_forces()[2]

print(f"Case 36: M = {M} k-ft at x = {a} ft, L = {L} ft")
print(f"R_A = {float(R_A):.3f} kips  (check: {-M/L:.3f})")
print(f"R_B = {float(R_B):.3f} kips  (check: {M/L:.3f})")

fig36 = plot_beam_full(beam36, I=beam_shape.I_x,
                       title=f"Case 36: Concentrated Moment at x={a} ft")
plt.show()

---
## Case 37: Continuous Beam — Two Spans, Different Lengths — Uniform Load on One Span

**AISC SCM Table 3-23, Case 37**

Lengths L1 and L2. Uniform load w on L1.
- M_2 = wL1³ / [8(L1 + L2)]
- R1 = wL1/2 - M_2/L1
- R2 = wL1/2 + M_2/L1 + M_2/L2
- R3 = -M_2/L2

In [ ]:
L1, L2 = 15.0, 25.0
w = 1.5

beam37 = ContinuousBeam(L1+L2, intermediate_supports=[L1])
beam37.pinned_support = 0
beam37.rolling_support = L1+L2
beam37.add_loads([DistributedLoadV(f'{w}', (0, L1))])

reactions = beam37.get_reaction_forces()
R1, R3 = reactions[1], reactions[2]
R2 = beam37._redundant_reactions[L1]

M2_formula = (w * L1**3) / (8 * (L1 + L2))
check1 = w*L1/2 - M2_formula/L1
check3 = -M2_formula/L2
check2 = w*L1/2 + M2_formula/L1 + M2_formula/L2

print(f"Reactions: R1={float(R1):.3f}, R2={float(R2):.3f}, R3={float(R3):.3f}")
print(f"Checks: R1={check1:.3f}, R2={check2:.3f}, R3={check3:.3f}")

fig37 = plot_beam_full(beam37, I=beam_shape.I_x, title='Case 37: Continuous Beam - Two Spans, Different Lengths - Uniform Load on One Span')
plt.show()

---
## Case 38: Continuous Beam — Two Spans, Different Lengths — Uniform Load on Both Spans

**AISC SCM Table 3-23, Case 38**

Lengths L1 and L2. Uniform load w on both spans.
- M_2 = w(L1³ + L2³) / [8(L1 + L2)]
- R1 = wL1/2 - M_2/L1
- R2 = w(L1 + L2)/2 + M_2/L1 + M_2/L2
- R3 = wL2/2 - M_2/L2

In [ ]:
L1, L2 = 15.0, 25.0
w = 1.2

beam38 = ContinuousBeam(L1+L2, intermediate_supports=[L1])
beam38.pinned_support = 0
beam38.rolling_support = L1+L2
beam38.add_loads([DistributedLoadV(f'{w}', (0, L1+L2))])

reactions = beam38.get_reaction_forces()
R1, R3 = reactions[1], reactions[2]
R2 = beam38._redundant_reactions[L1]

M2_formula = w * (L1**3 + L2**3) / (8 * (L1 + L2))
check1 = w*L1/2 - M2_formula/L1
check2 = w*(L1 + L2)/2 + M2_formula/L1 + M2_formula/L2
check3 = w*L2/2 - M2_formula/L2

print(f"Reactions: R1={float(R1):.3f}, R2={float(R2):.3f}, R3={float(R3):.3f}")
print(f"Checks: R1={check1:.3f}, R2={check2:.3f}, R3={check3:.3f}")

fig38 = plot_beam_full(beam38, I=beam_shape.I_x, title='Case 38: Continuous Beam - Two Spans, Different Lengths - Uniform Load on Both Spans')
plt.show()

---
## Case 39: Continuous Beam — Two Spans, Different Lengths — Concentrated Load on One Span

**AISC SCM Table 3-23, Case 39**

Lengths L1 and L2. Load P on L1 at distance a from R1 and b from R2.
- M_2 = Pab(L1 + a) / [2L1(L1 + L2)]
- R1 = Pb/L1 - M_2/L1
- R2 = Pa/L1 + M_2/L1 + M_2/L2
- R3 = -M_2/L2

In [ ]:
L1, L2 = 18.0, 12.0
a, b = 10.0, 8.0
P = 15.0

beam39 = ContinuousBeam(L1+L2, intermediate_supports=[L1])
beam39.pinned_support = 0
beam39.rolling_support = L1+L2
beam39.add_loads([PointLoadV(P, a)])

reactions = beam39.get_reaction_forces()
R1, R3 = reactions[1], reactions[2]
R2 = beam39._redundant_reactions[L1]

M2_formula = P*a*b*(L1+a) / (2*L1*(L1+L2))
check1 = P*b/L1 - M2_formula/L1
check3 = -M2_formula/L2
check2 = P*a/L1 + M2_formula/L1 + M2_formula/L2

print(f"Reactions: R1={float(R1):.3f}, R2={float(R2):.3f}, R3={float(R3):.3f}")
print(f"Checks: R1={check1:.3f}, R2={check2:.3f}, R3={check3:.3f}")

fig39 = plot_beam_full(beam39, I=beam_shape.I_x, title='Case 39: Continuous Beam - Two Spans, Different Lengths - Concentrated Load on One Span')
plt.show()

---
## Case 40: Continuous Beam — Three Equal Spans — Uniform Load on One End Span

**AISC SCM Table 3-23, Case 40**

Three spans of length L. Load w on first span.
- R1 = 0.450wL
- R2 = 0.550wL
- R3 = -0.050wL
- R4 = 0.050wL
- M_2 = 0.050wL²
- M_3 = -0.050wL²

In [ ]:
L = 15.0
w = 1.0

beam40 = ContinuousBeam(3*L, intermediate_supports=[L, 2*L])
beam40.pinned_support = 0
beam40.rolling_support = 3*L
beam40.add_loads([DistributedLoadV(f'{w}', (0, L))])

reactions = beam40.get_reaction_forces()
R1, R4 = reactions[1], reactions[2]
R2 = beam40._redundant_reactions[L]
R3 = beam40._redundant_reactions[2*L]

print(f"Reactions: R1={float(R1):.3f}, R2={float(R2):.3f}, R3={float(R3):.3f}, R4={float(R4):.3f}")
print(f"Checks: R1={0.450*w*L:.3f}, R2={0.550*w*L:.3f}, R3={-0.050*w*L:.3f}, R4={0.050*w*L:.3f}")

fig40 = plot_beam_full(beam40, I=beam_shape.I_x, title='Case 40: Continuous Beam - Three Equal Spans - Uniform Load on One End Span')
plt.show()

---
## Case 41: Continuous Beam — Three Equal Spans — Uniform Load on Two End Spans

**AISC SCM Table 3-23, Case 41**

Three spans of length L. Load w on first and third spans.
- R1 = R4 = 0.400wL
- R2 = R3 = 0.600wL
- M_2 = M_3 = 0.100wL²

In [ ]:
L = 15.0
w = 1.0

beam41 = ContinuousBeam(3*L, intermediate_supports=[L, 2*L])
beam41.pinned_support = 0
beam41.rolling_support = 3*L
beam41.add_loads([DistributedLoadV(f'{w}', (0, L)), DistributedLoadV(f'{w}', (2*L, 3*L))])

reactions = beam41.get_reaction_forces()
R1, R4 = reactions[1], reactions[2]
R2 = beam41._redundant_reactions[L]
R3 = beam41._redundant_reactions[2*L]

print(f"Reactions: R1={float(R1):.3f}, R2={float(R2):.3f}, R3={float(R3):.3f}, R4={float(R4):.3f}")
print(f"Checks: R1={0.400*w*L:.3f}, R2={0.600*w*L:.3f}, R3={0.600*w*L:.3f}, R4={0.400*w*L:.3f}")

fig41 = plot_beam_full(beam41, I=beam_shape.I_x, title='Case 41: Continuous Beam - Three Equal Spans - Uniform Load on Two End Spans')
plt.show()

---
## Case 42: Continuous Beam — Three Equal Spans — Uniform Load on All Spans

**AISC SCM Table 3-23, Case 42**

Three spans of length L. Load w on all spans.
- R1 = R4 = 0.400wL
- R2 = R3 = 1.100wL
- M_2 = M_3 = 0.100wL²

In [ ]:
L = 15.0
w = 1.0

beam42 = ContinuousBeam(3*L, intermediate_supports=[L, 2*L])
beam42.pinned_support = 0
beam42.rolling_support = 3*L
beam42.add_loads([DistributedLoadV(f'{w}', (0, 3*L))])

reactions = beam42.get_reaction_forces()
R1, R4 = reactions[1], reactions[2]
R2 = beam42._redundant_reactions[L]
R3 = beam42._redundant_reactions[2*L]

print(f"Reactions: R1={float(R1):.3f}, R2={float(R2):.3f}, R3={float(R3):.3f}, R4={float(R4):.3f}")
print(f"Checks: R1={0.400*w*L:.3f}, R2={1.100*w*L:.3f}, R3={1.100*w*L:.3f}, R4={0.400*w*L:.3f}")

fig42 = plot_beam_full(beam42, I=beam_shape.I_x, title='Case 42: Continuous Beam - Three Equal Spans - Uniform Load on All Spans')
plt.show()

### AISC Case 43: Simple Beam - Single Moving Load

**Reactions and Maxima:**
- $R_1 = P(L-x)/L$
- $R_2 = Px/L$
- $M_{max} = \frac{PL}{4}$ (when load is at center)
- $V_{max} = P$ (at ends)

In [ ]:
from civilpy.structural.beam_bending import Beam, PointLoadV, plot_beam_full
import numpy as np

L = 20
P = 10
beam = Beam(L)
beam.pinned_support = 0
beam.rolling_support = L

# Define moving load configuration (single load at offset 0)
moving_loads = [PointLoadV(P, 0)]

x_vec, max_v, min_v, max_m, min_m = beam.get_envelope(moving_loads, step_size=0.1)

print(f"Theoretical Max Moment: {P*L/4}")
print(f"Envelope Max Moment: {max(max_m):.2f}")
print(f"Theoretical Max Shear: {P}")
print(f"Envelope Max Shear: {max(max_v):.2f}")

beam.plot_envelope(x_vec, max_v, min_v, max_m, min_m)

### AISC Case 44: Simple Beam - Two Equal Moving Loads

**Reactions and Maxima:**
- $M_{max} = \frac{P}{2L}(L - a/2)^2$ (when $x = L/2 - a/4$)
- $V_{max} = P(2 - a/L)$ (at ends)

In [ ]:
L = 50
P = 10
a = 14
beam = Beam(L)
beam.pinned_support = 0
beam.rolling_support = L

# Two equal loads P separated by distance a
moving_loads = [PointLoadV(P, 0), PointLoadV(P, a)]

x_vec, max_v, min_v, max_m, min_m = beam.get_envelope(moving_loads, step_size=0.5)

theoretical_m = (P/(2*L)) * (L - a/2)**2
theoretical_v = P * (2 - a/L)

print(f"Theoretical Max Moment: {theoretical_m:.2f}")
print(f"Envelope Max Moment: {max(max_m):.2f}")
print(f"Theoretical Max Shear: {theoretical_v:.2f}")
print(f"Envelope Max Shear: {max(max_v):.2f}")

beam.plot_envelope(x_vec, max_v, min_v, max_m, min_m)

### AISC Case 45: Simple Beam - Two Unequal Moving Loads
This case demonstrates a real-world application using the AASHTO HS-20 truck model.

**Truck Configuration (HS-20):**
- Axle 1: 8 kips
- Axle 2: 32 kips (14 ft from Axle 1)
- Axle 3: 32 kips (14 ft from Axle 2)

In [ ]:
from civilpy.structural.aashto.vehicles import HS20Load

L = 100
beam = Beam(L)
beam.pinned_support = 0
beam.rolling_support = L

hs20 = HS20Load()
moving_loads = []
for k, axle in hs20.axles.items():
    if isinstance(k, int):
        moving_loads.append(PointLoadV(axle['load_kip'], axle['dist_ft']))

x_vec, max_v, min_v, max_m, min_m = beam.get_envelope(moving_loads, step_size=1.0)

print(f"Total Truck Weight: {hs20.total_axle_load_kip()} kips")
print(f"Envelope Max Moment: {max(max_m):.2f} kip-ft")
print(f"Envelope Max Shear: {max(max_v):.2f} kips")

beam.plot_envelope(x_vec, max_v, min_v, max_m, min_m)